In [ ]:
# Import required libraries
import re
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Import Dataset
df = pd.read_csv("E:\MJ-DEV\Github\Course-Materials\\2) Data PreProcessing\dataset\Employee-DataBase.csv")
df.head()

# Prep Work

In [ ]:
# Rename columnnames to small case
def column_cleansing(col):
    col = col.lower()
    col = re.sub("[^a-zA-Z0-9]+", "_", col)
    return col

print("Columns before transforming: ", df.columns)

df.columns = [ column_cleansing(col) for col in df.columns]
print("Columns after transforming: ", df.columns)

# Check Redundent Data

In [ ]:
# Print Number of duplicate records in dataframe
"""
At this Stage, Pandas not considering rows with null values, due to this you might not able to see full records. 
So this is not the place to perform this task."""
df.groupby(df.columns.tolist(), as_index=False).size()

# Handle Missing Data

**Step:1** *Identify Null (or) NaN values row wise*

In [ ]:
# df.shape[1] - df.count(axis=1)
df.isnull().sum(axis=1)

**Step:2** *Identify Null (or) NaN values column wise*

In [ ]:
# df.isnull().sum()
# df.count()
# for col in df.columns:
#     cnt = df[col].isnull().sum()
#     if cnt >= 1:
#         print(col, ': ', cnt)

**Step:3** *Identify Null (or) NaN values on whole dataframe with more info*

In [ ]:
df.info()

In [ ]:
# Replace Missing Values for integer object

df_target = df[["resignation"]]
df_obj = df[[col for col in df.columns if df[col].dtype == "object" and col not in df_target.columns]]
imp_obj = SimpleImputer(missing_values=np.NaN, strategy="most_frequent")
df_obj = pd.DataFrame(imp_obj.fit_transform(df_obj), columns=df_obj.columns)

# Check Empty data
# print(df_obj.isnull().sum())

df_num = df[[col for col in df.columns if df[col].dtype != "object" and col not in df_target.columns]]
imp_num = SimpleImputer(missing_values=np.NaN, strategy="mean")
df_num = pd.DataFrame(imp_num.fit_transform(df_num), columns=df_num.columns)

# Check Empty data
# print(df_num.isnull().sum())

df = pd.concat([df_obj, df_num, df_target], axis=1)

# Handle Categorical Data

In [ ]:
# Replace Categorical value with actual value: INDEPENDENT VARIABLE
cat_columns = ['designation', 'business_unit', 'gender', 'billable', 'hike']

df = pd.get_dummies(data=df, columns=cat_columns, drop_first=True)
df['resignation'] = df.pop('resignation')

# Replace catagorical value with actual value: DEPENDENT VARIABLE
df["resignation"] = lbl.fit_transform(df["resignation"])

df.head()

# Feature Selection - Stage: 1

In [ ]:
# Remove Redundent/Unwanted Information
df = df.drop(['empname', 'mail_id', 'mobile_number', 'on_boarded', 'empid', 'nationality'], axis=1)
df = df.drop_duplicates()
df.head()

# Normalizing dominant data

In [ ]:
sclr = StandardScaler()
df["salary"] = sclr.fit_transform(df["salary"].values.reshape(-1, 1))
df["age"] = sclr.fit_transform(df["age"].values.reshape(-1, 1))
df["hike_"] = sclr.fit_transform(df["hike_"].values.reshape(-1, 1))
df.head()

# Split dataframe to train and test sets

In [ ]:
# Create X and y Variable
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Split dataset to train and test sets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)